# Notebook 06: Finite Element Geometry Preparation & Non-Invasive Mesh Repair
**Project**: Stegoceras Biomechanics & Uncertainty Quantification  
**Specimen**: *Stegoceras validum* (UALVP 2, referred specimen)  
**Deliverable**: Phase 4 FE Geometry Preparation Pipeline  

### Objective
Perform non-invasive topological repair of the raw MorphoSource UALVP 2 skull mesh (`data/meshes/raw/witmerlab/stegoceras_ualvp2_skull.stl`), verifying volume conservation ($|\Delta V| < 0.05\%$) and surface fidelity ($|\Delta A| < 0.20\%$) using exact divergence-theorem surface integrals, and generating multi-tier tetrahedral meshes with zero inverted elements via TetGen.

In [ ]:
import numpy as np
import trimesh
from pathlib import Path
from stegoceras_biomechanics.fea.geometry import prepare_watertight_surface
from stegoceras_biomechanics.fea.meshing import generate_tetrahedral_mesh

raw_stl = Path('../data/meshes/original/whole_skull/WitmerLab_Stegoceras_UALVP2-000018284.stl')
clean_stl = Path('../data/meshes/cleaned/stegoceras_ualvp2_watertight.stl')

print('Executing non-invasive surface repair...')
clean_mesh, repair_report = prepare_watertight_surface(raw_stl, clean_stl)
print(f'Raw Volume: {repair_report.original_volume:,.1f} mm³ | Repaired Volume: {repair_report.repaired_volume:,.1f} mm³')
print(f'Volume Change: {repair_report.volume_change_pct:+.4f}% (Tolerance: ±0.05%)')
print(f'Area Change: {repair_report.area_change_pct:+.4f}% (Tolerance: ±0.20%)')
print(f'Max Surface Deviation: {repair_report.max_surface_deviation_mm:.3f} mm')
print(f'Watertight: {repair_report.repaired_watertight}')
assert repair_report.repaired_watertight, 'Mesh is not watertight!'
assert abs(repair_report.volume_change_pct) < 0.05, 'Volume change exceeds 0.05% tolerance!'
print('✓ Topological repair passed all geometric fidelity criteria!')

### Tetrahedral Mesh Generation & Quality Metrics
Inspect the generated solid tetrahedral meshes (Coarse and Medium tiers).

In [ ]:
coarse_data = np.load('../data/meshes/cleaned/stegoceras_tetmesh_coarse.npz')
med_data = np.load('../data/meshes/cleaned/stegoceras_tetmesh_medium.npz')

print('=== Coarse Mesh Quality ===')
print(f'Nodes: {len(coarse_data["nodes"]):,} | Elements: {len(coarse_data["elements"]):,}')
print(f'Total Volume: {np.sum(coarse_data["volumes"]):,.1f} mm³')
print(f'Min Aspect Ratio: {np.min(coarse_data["aspect_ratios"]):.2f} | Mean Aspect Ratio: {np.mean(coarse_data["aspect_ratios"]):.2f}')
print(f'Inverted Elements (V <= 0): {np.sum(coarse_data["volumes"] <= 0)}')
assert np.all(coarse_data['volumes'] > 0), 'Coarse mesh contains inverted elements!'

print('\n=== Medium Mesh Quality ===')
print(f'Nodes: {len(med_data["nodes"]):,} | Elements: {len(med_data["elements"]):,}')
print(f'Total Volume: {np.sum(med_data["volumes"]):,.1f} mm³')
print(f'Min Aspect Ratio: {np.min(med_data["aspect_ratios"]):.2f} | Mean Aspect Ratio: {np.mean(med_data["aspect_ratios"]):.2f}')
print(f'Inverted Elements (V <= 0): {np.sum(med_data["volumes"] <= 0)}')
assert np.all(med_data['volumes'] > 0), 'Medium mesh contains inverted elements!'
print('✓ All solid tetrahedral meshes verified strictly positive element Jacobians!')